In [ ]:
import os
from copy import deepcopy
import warnings

import numpy as np
import random
import torch
random_seed = 2025
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Replace "0" with the desired GPU device index
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from utils import calc_logit_norm, validate, extract_features, \
    filter_features, visualize_3d_features, add_gaussian_noise, \
    configure_model, entropy_loss, collect_params, adapt_model, \
    animate_3d_features_plotly, get_partial_dataloader, class_counts, \
    visualize_3d_features_plotly, get_reduced_features, animate_2d_features_plotly
from models import CNN3

In [ ]:
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),  # converts to tensor and scales image pixel values to [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # normalize using MNIST's mean and std
])

train_dataset = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=True, download=False, transform=transform)
test_dataset  = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=False, download=False, transform=transform)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)
TRAIN = False

if TRAIN:
    # Initialize the model, define loss function and optimizer
    model = CNN3().to('cuda')
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training loop
    num_epochs = 5
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images.to('cuda'))
            loss = criterion(outputs, labels.to('cuda'))
            
            # Backward pass and optimization
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        val_accuracy, val_loss = validate(model, test_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {running_loss/len(train_loader):.4f}, Val Loss: {val_loss:.4f}")
        
    torch.save(model, '/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model_3d.pth')

In [ ]:
model = torch.load('/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model_3d.pth')

In [ ]:
print(f"Clean Test Accuracy: {validate(model, test_loader)[0]: .2%}")

In [ ]:
features, feature_labels = extract_features(model, test_loader, 'fc2')
# filtered_features, filtered_labels = filter_features(features, np.array(feature_labels), selected_labels=[0,1,8])
# visualize_3d_features_plotly(filtered_features, filtered_labels)
clean_reduced_features, clean_labels, reducer = get_reduced_features(features, feature_labels, dimensions=2,
                                                                     n_neighbors=15)


In [ ]:
%matplotlib widget

In [ ]:
SEVERITY = 4

noisy_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: add_gaussian_noise(x, severity=SEVERITY)),
    transforms.Normalize((0.1307,), (0.3081,))
])

noisy_test_dataset = datasets.MNIST(root='/home/thilina/SSD2/thilina/datasets/mnist', train=False, download=False, transform=noisy_transform)
noisy_train_loader = DataLoader(noisy_test_dataset, batch_size=64, shuffle=True)
noisy_test_loader = DataLoader(noisy_test_dataset, batch_size=64, shuffle=False)

In [ ]:
noisy_features, noisy_feature_labels = extract_features(model, noisy_test_loader)
filtered_noisy_features, filtered_noisy_labels = filter_features(noisy_features, np.array(noisy_feature_labels), selected_labels=[0,1,8])
# visualize_3d_features_plotly(filtered_noisy_features, filtered_noisy_labels)

In [ ]:
# validate noisy data
zero_shot_accuracy, _ = validate(model, noisy_test_loader)
print(f"Zero Shot Accuracy: {zero_shot_accuracy: .2%}")

## Balanced Adaptation

In [ ]:
model = torch.load('/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model_3d.pth')
_, balanced_reduced_feature_list = adapt_model(model, noisy_train_loader, noisy_test_loader, reducer=reducer, feature_layer='fc2', log_frequency=500)
# animate_3d_features(reduced_feature_list, noisy_feature_labels, save_path='balanced_adaptation_3d.mp4')

In [ ]:
# animate_3d_features_plotly(balanced_reduced_feature_list, noisy_feature_labels)
animate_2d_features_plotly(balanced_reduced_feature_list, noisy_feature_labels)

In [ ]:
filtered_feature_frames = []
for feature_t in balanced_reduced_feature_list:
    filtered_feature_t, filtered_labels = filter_features(feature_t, np.array(noisy_feature_labels), selected_labels=[0, 1, 5])
    filtered_feature_frames.append(filtered_feature_t)

animate_3d_features_plotly(filtered_feature_frames, filtered_labels)

## Imbalanced Adaptation

### Imbalanced Dataloader

In [ ]:
from utils import generate_oversample_indices, class_counts
from torch.utils.data import Subset
from torch.utils.data.sampler import WeightedRandomSampler

imbalanced_test_loader = get_partial_dataloader(test_dataset, [5], final_samples=10000)

class_counts(imbalanced_test_loader)

In [ ]:
model = torch.load('/home/thilina/SSD2/thilina/test-time-adaptation-further_experiments/classification/simple_classifier/mnist_model_3d.pth')
_, imbalanced_reduced_feature_list = adapt_model(model, imbalanced_test_loader, noisy_test_loader, reducer=None, feature_layer='fc2', epochs=40)



In [ ]:
# noisy_features, noisy_feature_labels = extract_features(model, noisy_test_loader)
# filtered_noisy_features, filtered_noisy_labels = filter_features(noisy_features, np.array(noisy_feature_labels), selected_labels=[0, 1, 8])
# visualize_3d_features(filtered_noisy_features, filtered_noisy_labels)

In [ ]:
animate_3d_features_plotly(imbalanced_reduced_feature_list, noisy_feature_labels)
# animate_3d_features(reduced_feature_list, noisy_feature_labels, save_path='imbalance_adaptation_3d.mp4')

In [ ]:
filtered_feature_frames = []
for feature_t in imbalanced_reduced_feature_list:
    filtered_feature_t, filtered_labels = filter_features(feature_t, np.array(noisy_feature_labels), selected_labels=[0, 1, 5])
    filtered_feature_frames.append(filtered_feature_t)

animate_3d_features_plotly(filtered_feature_frames, filtered_labels)